# Análise Exploratória — Credit Risk Dataset

Dataset: [Credit Risk Dataset (Kaggle)](https://www.kaggle.com/datasets/laotse/credit-risk-dataset)

Este notebook realiza uma análise exploratória de dados (EDA) sobre o dataset de risco de crédito, com o objetivo de entender o perfil dos solicitantes, a distribuição de inadimplência (`loan_status`) e os principais fatores associados ao default.

## Dicionário de Dados

| Coluna | Descrição |
|---|---|
| `person_age` | Idade da pessoa |
| `person_income` | Renda anual |
| `person_home_ownership` | Situação de posse da moradia |
| `person_emp_length` | Tempo de emprego (em anos) |
| `loan_intent` | Finalidade do empréstimo |
| `loan_grade` | Grau de risco do empréstimo |
| `loan_amnt` | Valor do empréstimo |
| `loan_int_rate` | Taxa de juros |
| `loan_status` | Status do empréstimo (0 = adimplente, 1 = inadimplente/default) |
| `loan_percent_income` | Percentual da renda comprometido com o empréstimo |
| `cb_person_default_on_file` | Histórico de default (Y/N) |
| `cb_person_cred_hist_length` | Tempo de histórico de crédito (anos) |


In [ ]:
# Bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", None)


In [ ]:
DATA_PATH = "credit_risk_dataset.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


## 1. Visão Geral dos Dados

In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
# Checagem de valores ausentes
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0")


In [ ]:
# Checagem de duplicatas
print(f"Linhas duplicadas: {df.duplicated().sum()}")


## 2. Limpeza e Tratamento

Pontos conhecidos deste dataset:
- `person_emp_length` e `loan_int_rate` costumam ter valores ausentes.
- `person_age` pode conter outliers extremos (idades acima de 100 anos).
- `person_emp_length` também pode conter outliers (ex.: valores acima de 60 anos de emprego).


In [ ]:
df_clean = df.copy()

# Outliers óbvios de idade e tempo de emprego (erros de digitação/registro)
print("Idades > 100:", (df_clean["person_age"] > 100).sum())
print("Tempo de emprego > 60 anos:", (df_clean["person_emp_length"] > 60).sum())

df_clean = df_clean[df_clean["person_age"] <= 100]
df_clean = df_clean[(df_clean["person_emp_length"].isna()) | (df_clean["person_emp_length"] <= 60)]

print(f"Shape após remoção de outliers óbvios: {df_clean.shape}")


In [ ]:
# Tratamento de valores ausentes
# person_emp_length: preencher com a mediana
df_clean["person_emp_length"] = df_clean["person_emp_length"].fillna(df_clean["person_emp_length"].median())

# loan_int_rate: preencher com a mediana por loan_grade (grades semelhantes tendem a ter juros parecidos)
df_clean["loan_int_rate"] = df_clean.groupby("loan_grade")["loan_int_rate"].transform(
    lambda x: x.fillna(x.median())
)

df_clean.isnull().sum()


## 3. Distribuição da Variável-Alvo (`loan_status`)

In [ ]:
target_counts = df_clean["loan_status"].value_counts(normalize=True) * 100
print(target_counts)

fig, ax = plt.subplots()
sns.countplot(data=df_clean, x="loan_status", palette="Set2", ax=ax)
ax.set_xticklabels(["Adimplente (0)", "Default (1)"])
ax.set_title("Distribuição da Variável-Alvo: Loan Status")
ax.set_xlabel("")
ax.set_ylabel("Quantidade")
for p in ax.patches:
    ax.annotate(f"{p.get_height():,.0f}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.show()


**Observação:** este é um problema de classificação desbalanceado — a classe de default costuma representar cerca de 20% das observações. Isso é importante para a etapa de modelagem (se houver), pois métricas como acurácia podem enganar; recall, precisão e AUC-ROC são mais indicadas.

## 4. Análise Univariada — Variáveis Numéricas

In [ ]:
num_cols = ["person_age", "person_income", "person_emp_length", "loan_amnt",
            "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"]

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df_clean[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Distribuição: {col}")
axes[-1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(x=df_clean[col], ax=axes[i], color="lightcoral")
    axes[i].set_title(f"Boxplot: {col}")
axes[-1].axis("off")
plt.tight_layout()
plt.show()


## 5. Análise Univariada — Variáveis Categóricas

In [ ]:
cat_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    order = df_clean[col].value_counts().index
    sns.countplot(data=df_clean, y=col, order=order, palette="viridis", ax=axes[i])
    axes[i].set_title(f"Distribuição: {col}")
plt.tight_layout()
plt.show()


## 6. Análise Bivariada — Relação com o Default (`loan_status`)

In [ ]:
# Taxa de default por categoria
for col in cat_cols:
    rate = df_clean.groupby(col)["loan_status"].mean().sort_values(ascending=False) * 100
    print(f"\nTaxa de default por {col} (%):")
    print(rate.round(2))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    rate = df_clean.groupby(col)["loan_status"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.values, y=rate.index, palette="rocket", ax=axes[i])
    axes[i].set_title(f"Taxa de Default (%) por {col}")
    axes[i].set_xlabel("Taxa de Default (%)")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(data=df_clean, x="loan_status", y=col, palette="Set2", ax=axes[i])
    axes[i].set_title(f"{col} vs Loan Status")
    axes[i].set_xticklabels(["Adimplente", "Default"])
axes[-1].axis("off")
plt.tight_layout()
plt.show()


**Insights esperados:**
- Solicitantes com `loan_grade` pior (D, E, F, G) tendem a ter taxas de default muito mais altas.
- `loan_percent_income` alto (empréstimo compromete grande parte da renda) tende a se associar a maior inadimplência.
- Histórico de default anterior (`cb_person_default_on_file` = Y) costuma estar associado a maior taxa de default atual.
- Finalidades como `DEBTCONSOLIDATION` e `MEDICAL` costumam apresentar risco relativamente maior.


## 7. Correlação entre Variáveis Numéricas

In [ ]:
corr = df_clean[num_cols + ["loan_status"]].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Matriz de Correlação")
plt.show()


## 8. Análise Cruzada: Renda x Valor do Empréstimo x Default

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_clean.sample(min(3000, len(df_clean)), random_state=42),
    x="person_income", y="loan_amnt", hue="loan_status",
    palette={0: "steelblue", 1: "crimson"}, alpha=0.5
)
plt.title("Renda Anual vs Valor do Empréstimo (colorido por Default)")
plt.xlabel("Renda Anual")
plt.ylabel("Valor do Empréstimo")
plt.legend(title="Loan Status", labels=["Adimplente", "Default"])
plt.show()


In [ ]:
# Faixas etárias e taxa de default
bins = [18, 25, 35, 45, 55, 65, 100]
labels = ["18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
df_clean["age_group"] = pd.cut(df_clean["person_age"], bins=bins, labels=labels, right=True)

age_default = df_clean.groupby("age_group")["loan_status"].mean() * 100

plt.figure(figsize=(8, 5))
sns.barplot(x=age_default.index, y=age_default.values, palette="mako")
plt.title("Taxa de Default por Faixa Etária")
plt.ylabel("Taxa de Default (%)")
plt.xlabel("Faixa Etária")
plt.show()


## 9. Principais Conclusões

- A base é desbalanceada em relação ao `loan_status`, o que deve ser considerado em qualquer modelagem futura (ex.: uso de `class_weight`, oversampling/undersampling, ou métricas como AUC-ROC e F1).
- `loan_grade`, `loan_percent_income`, `loan_int_rate` e `cb_person_default_on_file` aparentam ser fortes indicadores de risco de default.
- Recomenda-se tratar outliers de `person_age` e `person_emp_length` antes de qualquer modelagem.
- Próximos passos sugeridos: engenharia de variáveis (ex.: razão dívida/renda), encoding de variáveis categóricas e treinamento de modelos de classificação (Regressão Logística, Random Forest, XGBoost) com validação cruzada.
